In [1]:
"""
c5_modules.py
=============
C.5: Representation-theoretic derivation of 2/10.

Strategy:
1. Build E over ℤ/9ℤ.
2. Compute all left ideals generated by single basis elements.
3. Classify ideals by size (order as a set).
4. Look for a natural 2-out-of-10 structure.


By Néstor E. Ramos


"""

import numpy as np
from itertools import product

MOD = 9
N = 9
FANO = [(0,1,2), (0,3,4), (0,5,6), (1,3,5), (1,4,6), (2,3,6), (2,4,5)]


def build(t_sq, t_left, t_right):
    M = np.zeros((N, N, N), dtype=int)
    for i in range(N):
        M[7, i, i] = 1
        M[i, 7, i] = 1
    for (a, b, c) in FANO:
        M[a, b, c] = 1
        M[b, c, a] = 1
        M[c, a, b] = 1
        M[b, a, c] = -1 % MOD
        M[c, b, a] = -1 % MOD
        M[a, c, b] = -1 % MOD
    for i in range(7):
        M[i, i, 7] = -1 % MOD
    M[8, 8, 7] = t_sq % MOD
    for i in range(7):
        M[8, i, i] = t_left % MOD
        M[i, 8, i] = t_right % MOD
    M[8, 7, 7] = t_left % MOD
    M[7, 8, 8] = 1
    return M


def left_mats(M):
    L = []
    for i in range(N):
        Lmat = np.zeros((N, N), dtype=int)
        for j in range(N):
            for k in range(N):
                Lmat[k, j] = M[i, j, k] % MOD
        L.append(Lmat)
    return L


def left_ideal(x, L, cap=200000):
    """Generate the left ideal generated by x."""
    ideal = set()
    stack = [tuple(int(v) % MOD for v in x)]
    while stack:
        v = stack.pop()
        if v in ideal:
            continue
        ideal.add(v)
        if len(ideal) > cap:
            return None
        vv = np.array(v, dtype=int)
        for Lmat in L:
            new_v = tuple(int(w) % MOD for w in (Lmat @ vv))
            if new_v not in ideal:
                stack.append(new_v)
    return ideal


def is_power_of_9(n):
    """Check if n = 9^k for some k."""
    k = 0
    while 9 ** k < n:
        k += 1
    return 9 ** k == n, k


def analyze(name, M):
    L = left_mats(M)
    print("=" * 72)
    print(f"MODULE ANALYSIS: {name}")
    print("=" * 72)
    print()

    print("Left ideals generated by single basis elements:")
    print(f"{'element':>10} {'|ideal|':>10} {'= 9^k?':>12} {'dim?':>8}")
    print("-" * 50)

    for i in range(N):
        x = np.zeros(N, dtype=int); x[i] = 1
        ideal = left_ideal(x, L)
        if ideal is None:
            print(f"  e_{i:>2}:      > cap")
            continue
        n = len(ideal)
        is_pow, k = is_power_of_9(n)
        print(f"  e_{i:>2}:      {n:>10} {'yes' if is_pow else 'no':>12} {k if is_pow else '?':>8}")

    print()

    # Try the canonical electron elements
    candidates = {
        "e_0 (Fano unit)":       np.array([1,0,0,0,0,0,0,0,0]),
        "1 (identity)":          np.array([0,0,0,0,0,0,0,1,0]),
        "t (TRB)":               np.array([0,0,0,0,0,0,0,0,1]),
        "1 + t":                 np.array([0,0,0,0,0,0,0,1,1]),
        "1 - t":                 np.array([0,0,0,0,0,0,0,1,-1 % MOD]),
        "e_0 + e_1":             np.array([1,1,0,0,0,0,0,0,0]),
        "e_0 + 1":               np.array([1,0,0,0,0,0,0,1,0]),
        "e_0 + t":               np.array([1,0,0,0,0,0,0,0,1]),
    }

    print("Left ideals for electron-related elements:")
    print(f"{'element':>20} {'|ideal|':>10} {'= 9^k?':>10} {'k':>5}")
    print("-" * 50)

    results = {}
    for label, x in candidates.items():
        ideal = left_ideal(x, L, cap=100000)
        if ideal is None:
            print(f"  {label:>20}  > cap")
            continue
        n = len(ideal)
        is_pow, k = is_power_of_9(n)
        print(f"  {label:>20}  {n:>10} {'yes' if is_pow else 'no':>10} {k if is_pow else '?':>5}")
        results[label] = (n, k if is_pow else None)

    print()

    # Check the 8-dim ideal hypothesis: span{1, e_0..e_6}
    # This is a specific subspace. Is it a left ideal?
    e8 = np.zeros(N, dtype=int)
    for i in range(7):
        e8[i] = 1
    e8[7] = 1

    # Check closure of span{1, e_0, ..., e_6}
    basis_8 = [np.eye(N, dtype=int)[i] for i in range(7)] + [np.eye(N, dtype=int)[7]]
    closed_8 = True
    for Lmat in L:
        for b in basis_8:
            new = (Lmat @ b) % MOD
            # Check if new is in span of basis_8
            # (quick check: is component 8 zero?)
            if new[8] != 0:
                closed_8 = False
                break
        if not closed_8:
            break

    print(f"Span{{1, e_0..e_6}} (8-dim) is a left ideal? {closed_8}")
    print()

    # If closed, check the quotient
    if closed_8:
        print(f"  The quotient E / span{{1, e_0..e_6}} is 1-dimensional.")
        print(f"  It's spanned by [t] (the TRB element class).")
        print()

    # Explore the maximal left ideals
    print("Enumerating left ideals of small size (< 1000 vectors):")
    small_ideals = {}
    for x_int in range(MOD ** N):
        # Too many; sample instead
        pass

    # Sample many random vectors
    rng = np.random.default_rng(42)
    samples = 5000
    sizes_seen = {}
    for _ in range(samples):
        x = rng.integers(0, MOD, size=N)
        if np.all(x == 0):
            continue
        ideal = left_ideal(x, L, cap=5000)
        if ideal is None:
            continue
        n = len(ideal)
        is_pow, k = is_power_of_9(n)
        key = (n, k if is_pow else "non-pow")
        sizes_seen[key] = sizes_seen.get(key, 0) + 1

    print(f"  Sampled {samples} random vectors.")
    print(f"  Distinct left-ideal sizes encountered:")
    for key, count in sorted(sizes_seen.items(), key=lambda x: -x[1])[:15]:
        n, k = key
        print(f"    size {n:>8} ({'9^'+str(k) if isinstance(k,int) else k}) — {count} generators")

    print()
    return results


M_E = build(t_sq=0, t_left=3, t_right=-3)
M_G = build(t_sq=-1, t_left=1, t_right=-1)

results_E = analyze("E (drain)", M_E)
results_G = analyze("G (oscillator)", M_G)


# ============================================================
# 2/10 HYPOTHESIS TEST
# ============================================================
print("=" * 72)
print("THE 2/10 HYPOTHESIS")
print("=" * 72)
print()
print("Alphabet size = 10 (states 0-9). Algebra E has dimension 9.")
print("The electron 'uses' 8 slots (7 Fano + identity), leaving 2 free.")
print("Sub-channel = (10 - 8)/10 = 2/10 = 0.200.")
print()
print("Module-theoretic version:")
print("  The regular representation of E is 9-dimensional.")
print("  If the 'Fano + identity' subspace is an 8-dim left ideal I_8,")
print("  the complement is 1-dimensional (spanned by t).")
print("  So the module decomposition gives 8 + 1 = 9, not 8 + 2.")
print()
print("There is no 2-dimensional complement in the 9-dim algebra.")
print("The '2' in 2/10 must come from the 10-state alphabet,")
print("which includes the reserved state 9 (not an algebra element).")
print()
print("CONCLUSION:")
print("  The 2/10 fraction is NOT a module-theoretic invariant of E alone.")
print("  It requires the 10-state alphabet (including the reserved state).")
print("  Module theory does not uniquely determine it.")
print()


# ============================================================
# FINAL VERDICT
# ============================================================
print("=" * 72)
print("FINAL VERDICT ON C.5")
print("=" * 72)
print()
print("The module structure of E does NOT contain a 2-dim sub-module")
print("that would explain 2/10 as a theorem.")
print()
print("The algebra E decomposes into an 8-dim ideal (Fano + identity)")
print("and a 1-dim complement (t). The '2' of the sub-channel is not")
print("a module dimension; it comes from the physical alphabet size.")
print()
print("C.5 is a NEGATIVE result: module theory does not close the gap.")
print("The sub-channel 2/10 remains a physical/counting phenomenon,")
print("not a representation-theoretic theorem.")
print()

MODULE ANALYSIS: E (drain)

Left ideals generated by single basis elements:
   element    |ideal|       = 9^k?     dim?
--------------------------------------------------
  e_ 0:              37           no        ?
  e_ 1:              37           no        ?
  e_ 2:              37           no        ?
  e_ 3:              37           no        ?
  e_ 4:              37           no        ?
  e_ 5:              37           no        ?
  e_ 6:              37           no        ?
  e_ 7:              37           no        ?
  e_ 8:              20           no        ?

Left ideals for electron-related elements:
             element    |ideal|     = 9^k?     k
--------------------------------------------------
       e_0 (Fano unit)          37         no     ?
          1 (identity)          37         no     ?
               t (TRB)          20         no     ?
                 1 + t          39         no     ?
                 1 - t          39         no     ?
           